### Aggregation with Station Groups (3-Hour Bins)

In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
code_lookup = pd.read_csv('../Project_datasets/station_code_lookup.csv')
print(f"Lookup shape: {code_lookup.shape}")
code_lookup.head()

In [ ]:
# Build dictionaries: year -> set of station codes for each group
mont_royal_codes = {}
berri_codes = {}

for year in range(2014, 2018):
    mont_royal_codes[year] = set(
        code_lookup[(code_lookup['group'] == 'mont_royal') & (code_lookup['year'] == year)]['station_code']
    )
    berri_codes[year] = set(
        code_lookup[(code_lookup['group'] == 'berri') & (code_lookup['year'] == year)]['station_code']
    )

# Print code sets for verification
for year in range(2014, 2018):
    print(f"{year}: Mont-Royal codes = {mont_royal_codes[year]}")
    print(f"      Berri codes = {berri_codes[year]}")

#### Process OD files one at a time

In [ ]:
hourly_chunks = []

for year in range(2014, 2018):
    for month in range(4, 12):  # April to November
        file_name = f"OD_{year}-{month:02d}.csv"
        file_path = os.path.join('../Data', file_name)
        
        if not os.path.exists(file_path):
            print(f"  Skipping {file_name} (not found)")
            continue
        
        # Load single file
        df = pd.read_csv(file_path)
        
        # Filter: start_station in Mont-Royal group AND end_station in Berri group
        mr_codes = mont_royal_codes[year]
        br_codes = berri_codes[year]
        
        mask = (
            df['start_station_code'].isin(mr_codes) & 
            df['end_station_code'].isin(br_codes)
        )
        filtered = df[mask].copy()
        
        if len(filtered) == 0:
            print(f"  {file_name}: 0 matching rides")
            continue
        
        # Parse timestamps, floor to 3-hour bin
        filtered['start_date'] = pd.to_datetime(filtered['start_date'])
        filtered['time_bin'] = filtered['start_date'].dt.floor('3h')
        
        # Aggregate per 3-hour bin
        binned = filtered.groupby('time_bin').agg(
            trip_count=('duration_sec', 'count'),
            avg_duration=('duration_sec', 'mean'),
            member_count=('is_member', 'sum')
        ).reset_index()
        
        hourly_chunks.append(binned)
        print(f"  {file_name}: {len(filtered)} rides -> {len(binned)} time bins")

print(f"\nTotal chunks: {len(hourly_chunks)}")

In [ ]:
raw = pd.concat(hourly_chunks, ignore_index=True)

# Re-aggregate in case of overlaps between monthly files
combined = raw.groupby('time_bin').agg(
    trip_count=('trip_count', 'sum'),
    avg_duration=('avg_duration', 'mean'),
    member_count=('member_count', 'sum')
).reset_index()

print(f"Combined data: {len(combined)} rows")
print(f"Date range: {combined['time_bin'].min()} to {combined['time_bin'].max()}")

#### Create complete 3-hour time index

In [ ]:
# Create complete 3-hour index for each season (April 1 to November 30, each year)
full_index_parts = []
for year in range(2014, 2018):
    start = pd.Timestamp(f'{year}-04-01 00:00:00')
    end = pd.Timestamp(f'{year}-11-30 21:00:00')  # last 3h bin of the day starts at 21:00
    bins = pd.date_range(start, end, freq='3h')
    full_index_parts.append(bins)

full_index = pd.DatetimeIndex(sorted(set().union(*[set(idx) for idx in full_index_parts])))
full_df = pd.DataFrame({'time_bin': full_index})

# Merge with observed data
df_full = full_df.merge(combined, on='time_bin', how='left')

# Fill missing bins with 0 trips
df_full['trip_count'] = df_full['trip_count'].fillna(0).astype(int)
df_full['member_count'] = df_full['member_count'].fillna(0).astype(int)
df_full['avg_duration'] = df_full['avg_duration'].fillna(0)

print(f"Complete dataset: {len(df_full)} rows")
print(f"Expected: ~{4 * 244 * 8} rows (4 years x ~244 days x 8 bins/day)")

In [ ]:
print("Trip count distribution:")
print(df_full['trip_count'].describe())
print(f"\nZero-trip bins: {(df_full['trip_count'] == 0).sum()} ({(df_full['trip_count'] == 0).mean():.1%})")
print(f"Non-zero bins: {(df_full['trip_count'] > 0).sum()}")
print(f"\nTrips per year:")
df_full['year'] = df_full['time_bin'].dt.year
print(df_full.groupby('year')['trip_count'].agg(['sum', 'mean', 'count']))
df_full.drop(columns=['year'], inplace=True)

In [ ]:
df_full.to_csv('../Project_datasets/hourly_grouped_rides.csv', index=False)
print(f"Saved hourly_grouped_rides.csv ({len(df_full)} rows)")

In [ ]:
# Quick sanity check
print(f"Total trips: {df_full['trip_count'].sum()}")
print(f"Avg trips/bin (non-zero): {df_full[df_full['trip_count'] > 0]['trip_count'].mean():.2f}")
print(f"Max trips in one bin: {df_full['trip_count'].max()}")
print(f"\nFirst few rows:")
df_full.head(10)